### Limpieza del fichero 05-estaciones-acusticas.csv. Se carga desde Kaggle y se limpia y normalizan los datos

In [1]:
!pip install reverse_geocoder

In [1]:
import pandas as pd
import reverse_geocoder as rg
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.decomposition import PCA
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np



pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
file_path_05 = "05-estaciones-acusticas.csv"

df_05 = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "raquelahdo/ruido-datasets",
  file_path_05,
    pandas_kwargs={"encoding": "latin-1", "sep": ";"}
)

In [3]:
df_05.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ESTACIÓN    31 non-null     object
 1   NOMBRE      31 non-null     object
 2   UBICACIÓN   31 non-null     object
 3   DISTRITO    31 non-null     object
 4   BARRIO      31 non-null     object
 5   LONGITUD    31 non-null     object
 6   LATITUD     31 non-null     object
 7   X           31 non-null     int64 
 8   Y           31 non-null     int64 
 9   ALTITUD     31 non-null     int64 
 10  FECHA ALTA  31 non-null     object
dtypes: int64(3), object(8)
memory usage: 2.8+ KB


In [4]:
df_05.describe()

,X,Y,ALTITUD
count,31.000000,3.100000e+01,31.000000
mean,442359.387097,4.475713e+06,657.354839
std,4516.280340,4.528396e+03,35.726319
min,433969.000000,4.466527e+06,601.000000
25%,439634.000000,4.472704e+06,626.000000
50%,441564.000000,4.475063e+06,657.000000
75%,444979.000000,4.478992e+06,681.500000
max,450829.000000,4.485540e+06,728.000000


In [5]:
missing_values_05 = df_05.isnull().sum()

display(missing_values_05)

ESTACIÓN      0
NOMBRE        0
UBICACIÓN     0
DISTRITO      0
BARRIO        0
LONGITUD      0
LATITUD       0
X             0
Y             0
ALTITUD       0
FECHA ALTA    0
dtype: int64

In [6]:
df_05.head()

,ESTACIÓN,NOMBRE,UBICACIÓN,DISTRITO,BARRIO,LONGITUD,LATITUD,X,Y,ALTITUD,FECHA ALTA
0,RF-01,Paseo de Recoletos,Frente al n23 del Paseo de Recoletos,Centro,Justicia,-3.691.877,40.422.599,441307,4474893,648,07/03/2011
1,RF-02,Carlos V,"Plaza del Emperador Carlos V, junto al n1 del ...",Retiro,Jerónimos,-3.691.509,40.409.121,441327,4473397,629,01/12/1998
2,RF-03,Plaza del Carmen,Plaza del Carmen frente al n3 de la Calle de l...,Centro,Sol,-3.703.175,40.419.251,440346,4474529,657,01/11/1999
3,RF-04,Plaza de España,Frente al n18 de la Plaza de España,Moncloa - Aravaca,Argüelles,-3.712.253,40.424.005,439580,4475063,637,01/12/1998
4,RF-05,Barrio del Pilar,Parque de La Vaguada semiesquina de Avenida Mo...,Fuencarral - El Pardo,Pilar,-3.711.543,40.478.197,439688,4481078,673,07/06/1999


Renombrado estandarizado de columnas: Nombres en minusculas, sin espacios, sin acentos, consistentes con Python.
Convertir tipos numéricos correctamente (a float)
Creación de columna Fecha.
Ordenar dataset por fecha.

Dificultades del dataset original:
A) LONGITUD y LATITUD vienen como texto con puntos de millar. 
Eliminamos puntos intermedios
Convertimos a float
Preservamos signo negativo
Ejemplo:
-3.691.877 → debe ser -3.691877
40.422.599 → debe ser 40.422599
B) FECHA ALTA está en formato DD/MM/AAAA, no datetime.
C) Nombres de columnas con acentos y mayúsculas.
D) X, Y, ALTITUD deben ser numéricos.
E) UBICACIÓN, DISTRITO, BARRIO contienen texto largo → mantener como string.

In [7]:
#creamos una columna fecha para trabajar
#renombramos las columnas es formato snake_case, sin acentos, consistente con los datos trabajados.

df_05 = df_05.rename(columns={
    "ESTACIÓN": "estacion",
    "NOMBRE": "nombre",
    "UBICACIÓN": "ubicacion",
    "DISTRITO": "distrito",
    "BARRIO": "barrio",
    "LONGITUD": "longitud",
    "LATITUD": "latitud",
    "X": "utm_x",
    "Y": "utm_y",
    "ALTITUD": "altitud",
    "FECHA ALTA": "fecha_alta"
})

#Limpieza de coordenadas de latitud y longitud
for col in ["longitud", "latitud"]:
    df_05[col] = (
        df_05[col]
        .astype(str)
        .str.replace(".", "", regex=False)      # elimina separador de miles
        .str.replace(",", ".", regex=False)     # por si hubiera comas
        .astype(float) / 1e5                    # convertimos a decimal real
    )

df_05["longitud"] = df_05["longitud"].astype(str).str.replace(".", "", regex=False).astype(float) / 1e6
df_05["latitud"] = df_05["latitud"].astype(str).str.replace(".", "", regex=False).astype(float) / 1e6


#Conversión de coordenadas UTM y altitud
df_05["utm_x"] = pd.to_numeric(df_05["utm_x"], errors="coerce")
df_05["utm_y"] = pd.to_numeric(df_05["utm_y"], errors="coerce")
df_05["altitud"] = pd.to_numeric(df_05["altitud"], errors="coerce")

# Conversión de FECHA ALTA a datetime
df_05["fecha_alta"] = pd.to_datetime(
    df_05["fecha_alta"],
    format="%d/%m/%Y",
    errors="coerce"
)

#Limpieza de strings: quitar espacios extra y normalizar
for col in ["estacion", "nombre", "ubicacion", "distrito", "barrio"]:
    df_05[col] = df_05[col].astype(str).str.strip()

#Ordenar por estación
df_05 = df_05.sort_values("estacion").reset_index(drop=True)


In [8]:
df_05.head()

,estacion,nombre,ubicacion,distrito,barrio,longitud,latitud,utm_x,utm_y,altitud,fecha_alta
0,RF-01,Paseo de Recoletos,Frente al n23 del Paseo de Recoletos,Centro,Justicia,-3.691877,40.422599,441307,4474893,648,2011-03-07
1,RF-02,Carlos V,"Plaza del Emperador Carlos V, junto al n1 del ...",Retiro,Jerónimos,-3.691509,40.409121,441327,4473397,629,1998-12-01
2,RF-03,Plaza del Carmen,Plaza del Carmen frente al n3 de la Calle de l...,Centro,Sol,-3.703175,40.419251,440346,4474529,657,1999-11-01
3,RF-04,Plaza de España,Frente al n18 de la Plaza de España,Moncloa - Aravaca,Argüelles,-3.712253,40.424005,439580,4475063,637,1998-12-01
4,RF-05,Barrio del Pilar,Parque de La Vaguada semiesquina de Avenida Mo...,Fuencarral - El Pardo,Pilar,-3.711543,40.478197,439688,4481078,673,1999-06-07


In [9]:
df_05.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   estacion    31 non-null     object        
 1   nombre      31 non-null     object        
 2   ubicacion   31 non-null     object        
 3   distrito    31 non-null     object        
 4   barrio      31 non-null     object        
 5   longitud    31 non-null     float64       
 6   latitud     31 non-null     float64       
 7   utm_x       31 non-null     int64         
 8   utm_y       31 non-null     int64         
 9   altitud     31 non-null     int64         
 10  fecha_alta  31 non-null     datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 2.8+ KB


//Para este fichero no es necesario habilitar la gestión masiva de datos//


NECESIDAD DE HABILITAR LA GESTIÓN MASIVA DE DATOS
The number of rows in your dataset is greater than the maximum allowed (5000).

Try enabling the VegaFusion data transformer which raises this limit by pre-evaluating data
transformations in Python.
    >> import altair as alt
    >> alt.data_transformers.enable("vegafusion")

In [ ]:
#Se activa VegaFusion
import altair as alt
alt.data_transformers.enable("vegafusion")


In [ ]:
pip install "vegafusion[embed]>=1.5.0"

In [ ]:
 pip install "vl-convert-python>=1.6.0"

In [11]:

import altair as alt

# Desactivar motores que intentan usar vegafusion
alt.renderers.set_embed_options(actions=False)
alt.data_transformers.enable


<bound method PluginRegistry.enable of DataTransformerRegistry(active='default', registered=['csv', 'default', 'json', 'vegafusion'])>

In [12]:
#VISUALIZAVIÓN 1: Número de estaciones por distrito (barras)

chart_distritos = alt.Chart(df_05).mark_bar().encode(
    x=alt.X('distrito:N', sort='-y', title='Distrito'),
    y=alt.Y('count():Q', title='Número de estaciones'),
    color='distrito:N',
    tooltip=['distrito:N', 'count()']
).properties(
    title='Número de estaciones acústicas por distrito',
    width=600,
    height=400
)

chart_distritos


alt.Chart(...)

In [14]:
#VISUALIZACIÓN 2: Altitud por estación (bar chart ordenado)

chart_altitud = alt.Chart(df_05).mark_bar().encode(
    x=alt.X('estacion:N', sort='-y', title='Estación'),
    y=alt.Y('altitud:Q', title='Altitud (m)'),
    color='altitud:Q',
    tooltip=['estacion', 'nombre', 'altitud']
).properties(
    title='Altitud de cada estación acústica',
    width=700,
    height=400
)

chart_altitud



alt.Chart(...)

In [15]:
#VISUALIZACIÓN 3 - Estaciones activas a lo largo del tiempo (línea)

#Muestra cómo ha crecido la red:
df_timeline = df_05.groupby('fecha_alta', as_index=False).size()
df_timeline['acumulado'] = df_timeline['size'].cumsum()

chart_timeline = alt.Chart(df_timeline).mark_line(point=True).encode(
    x='fecha_alta:T',
    y='acumulado:Q',
    tooltip=['fecha_alta:T','acumulado:Q']
).properties(
    title='Evolución histórica del número de estaciones activas',
    width=700,
    height=300
)

chart_timeline

alt.Chart(...)

In [16]:
#VISUALIZACIÓN 4: Mapa de Madrid con las estaciones (Altair) - REVISAR NO OK

chart_mapa = alt.Chart(df_05).mark_circle(size=150).encode(
    longitude='longitud:Q',
    latitude='latitud:Q',
    color=alt.Color('distrito:N', title='Distrito'),
    tooltip=['estacion','nombre','distrito','barrio','latitud','longitud']
).properties(
    title='Mapa de estaciones acústicas de Madrid (WGS84)',
    width=500,
    height=500
)

chart_mapa


alt.Chart(...)

In [21]:
#EJEMPLO DE CÓDIGO ALTair PARA CARGAR Y PINTAR el GEOJSON - no funciona OK el mapa

#Guardar el archivo GEOJSON con encoding UTF‑8
import requests
import json

url_geojson = "https://geoportal.madrid.es/fsdescargas/IDEAM_WBGEOPORTAL/LIMITES_ADMINISTRATIVOS/Distritos/TopoJSON/Distritos.json"

r = requests.get(url_geojson)
r.encoding = "utf-8"  # Forzar codificación

with open("Distritos.json", "w", encoding="utf-8") as f:
    f.write(r.text)

with open("Distritos.json", "r", encoding="utf-8") as f:
    distritos_geo = json.load(f)


chart_distritos = alt.Chart(url_geojson).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).encode(
    color=alt.value("#e5e7eb")
).properties(
    width=500,
    height=500
)

chart_estaciones = alt.Chart(df_05).mark_circle(
    size=80,
    color='red'
).encode(
    longitude='longitud:Q',
    latitude='latitud:Q',
    tooltip=['estacion','nombre','distrito','barrio']
)

chart_distritos + chart_estaciones



alt.LayerChart(...)

In [22]:
#VISUALIZACIÓN 5: Selector interactivo de distrito + mapa filtrado

selector_distrito = alt.param(
    name='select_dist',
    bind=alt.binding_select(
        options=sorted(df_05['distrito'].unique().tolist()),
        name='Distrito: '
    ),
    value=sorted(df_05['distrito'].unique())[0]
)

chart_mapa_interactivo = alt.Chart(df_05).mark_circle(size=150).encode(
    longitude='longitud:Q',
    latitude='latitud:Q',
    color='barrio:N',
    tooltip=['estacion','nombre','barrio','distrito']
).add_params(
    selector_distrito
).transform_filter(
    selector_distrito
).properties(
    title='Mapa interactivo de estaciones por distrito',
    width=500,
    height=500
)

chart_mapa_interactivo


alt.Chart(...)

In [24]:
#VISUALIZACIÓN 6 — Estaciones por barrio (treemap opcional)


chart_barrio = alt.Chart(df_05).mark_bar().encode(
    x=alt.X('barrio:N', sort='-y'),
    y='count():Q',
    color='distrito:N',
    tooltip=['barrio','distrito','count()']
).properties(
    title='Estaciones acústicas por barrio',
    width=700,
    height=400
)

chart_barrio


alt.Chart(...)

En este dataset podemos correlacionar:

Altitud ↔ Latitud/Longitud (patrones topográficos)
Altitud ↔ Distritos
Latitud ↔ Longitud (para validar distribución espacial)

Aunque df_05 no tiene datos acústicos, estas correlaciones son relevantes para relación geográfica.

In [25]:
#MATRIZ DE CORRELACIÓN (longitud, latitud, UTM, altitud)
cols = ['longitud','latitud','utm_x','utm_y','altitud']
corr = df_05[cols].corr().reset_index().melt('index')

chart_corr = alt.Chart(corr).mark_rect().encode(
    x='index:N',
    y='variable:N',
    color=alt.Color('value:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['index','variable','value']
).properties(
    title='Matriz de correlación geográfica de estaciones',
    width=400,
    height=400
)

chart_corr



alt.Chart(...)

In [26]:
#DISPERSIÓN: Altitud vs Latitud 

chart_alt_lat = alt.Chart(df_05).mark_circle(size=60).encode(
    x='latitud:Q',
    y='altitud:Q',
    color='distrito:N',
    tooltip=['estacion','nombre','distrito','altitud']
).properties(
    title='Relación entre altitud y latitud',
    width=500,
    height=350
)

chart_alt_lat


alt.Chart(...)

In [27]:
#DISPERSIÓN: Altitud vs Longitud
chart_alt_lon = alt.Chart(df_05).mark_circle(size=60).encode(
    x='longitud:Q',
    y='altitud:Q',
    color='distrito:N',
    tooltip=['estacion','nombre','distrito','altitud']
).properties(
    title='Relación entre altitud y longitud',
    width=500,
    height=350
)

chart_alt_lon


alt.Chart(...)